In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix

# download dataset
path = kagglehub.dataset_download("kamilpytlak/personal-key-indicators-of-heart-disease")
print("File Path:", path)

# check its csv file
csv_file_path = None
for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            csv_file_path = os.path.join(root, file)
            break
    if csv_file_path:
        break

if not csv_file_path:
    raise ValueError("There is no CSV!")

print("CSV File is:", csv_file_path)

# Data to DataFrame
df = pd.read_csv(csv_file_path)
print("\n First 5 Line")
display(df.head())

print("\n Missing Values:\n", df.isnull().sum().max())

print("\n Class Distrubition (HeartDisease %):")
print(df['HeartDisease'].value_counts(normalize=True) * 100)

# Boxplot for BMI and Sleep Time outliers
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(y=df['BMI'], color='lightblue')
plt.title('BMI Outliers Control')

plt.subplot(1, 2, 2)
sns.boxplot(y=df['SleepTime'], color='lightgreen')
plt.title('Sleeping Time Outliers Control')

plt.tight_layout()
plt.show()

# EDA
# Eelationship between categorical variables and heart disease
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(x='Smoking', hue='HeartDisease', data=df, ax=axes[0])
axes[0].set_title('Sigara and Heart Disease')

sns.countplot(x='AlcoholDrinking', hue='HeartDisease', data=df, ax=axes[1])
axes[1].set_title('Alcohol Drinking and Heart Disease')

sns.countplot(x='PhysicalActivity', hue='HeartDisease', data=df, ax=axes[2])
axes[2].set_title('Physical Activity and Heart Disease')

plt.tight_layout()
plt.show()

# Distribution by age category
plt.figure(figsize=(10, 5))
sns.countplot(x='AgeCategory', hue='HeartDisease', data=df, order=sorted(df['AgeCategory'].unique()))
plt.title('Distribution of Heart Disease by Age category')
plt.xticks(rotation=45)
plt.show()

# Preprocessing and Partitioning
categorical_cols = df.select_dtypes(include=['object', 'category']).columns

le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

# %80 Train, %20 Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Scale numerical data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model Training (Lojistik Regression)
print("\n Model Training ")
baseline_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
baseline_model.fit(X_train_scaled, y_train)

# Predictions on the test set
y_pred = baseline_model.predict(X_test_scaled)
y_pred_proba = baseline_model.predict_proba(X_test_scaled)[:, 1]

# Results
print("\n Baseline Model Results (Lojistik Regression)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_proba))
print("\n Classification Report:\n")
print(classification_report(y_test, y_pred))

# Confusion Matrix
plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Real Values')
plt.xlabel('Prediction Values')
plt.show()